In [1]:
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [9]:
!ollama pull llama3.2

pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest 
pulling dde5aa3fc5ff... 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 2.0 GB                         
pulling 966de95ca8a6... 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 1.4 KB                         
pulling fcc5a6bec9da... 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 7.7 KB                         
pulling a70ff7e570d9... 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 6.0 KB                         
pulling 56bb8bd477a5... 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�   96 B                         
pulling 34bb5ab01051... 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [10]:
import requests
import json
import gradio as gr

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

# Initialize the OpenAI client for Ollama integration
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [18]:
import re

In [25]:
# import json
import re
from openai import OpenAI

# 🛠️ Llama 3 API Configuration
OLLAMA_API = "http://localhost:11434/v1"  # Ensure your Ollama server is running
MODEL = "llama3"  # Using Llama 3 (not 3.2)

# 🔧 Initialize OpenAI client for Llama 3 via Ollama
ollama_client = OpenAI(base_url=OLLAMA_API, api_key="ollama")

def fix_json_structure(response_text):
    """
    Fixes minor JSON formatting issues from Llama responses.
    """
    response_text = response_text.strip("```json").strip("```").strip()  # Remove Markdown formatting
    
    # 🔍 Extract JSON if mixed with other text
    match = re.search(r"\[\s*{.*}\s*\]", response_text, re.DOTALL)
    if match:
        response_text = match.group(0)  # Extract JSON content

    # 🔧 Remove trailing commas before closing brackets
    response_text = re.sub(r',\s*\]', ']', response_text)
    
    return response_text

def generate_mcqs(skills):
    """
    Generates 10 multiple-choice questions (MCQs) using Llama 3.
    """
    prompt = f"""
    Generate exactly 10 multiple-choice questions (MCQs) related to {skills}.
    Each question must have 4 options and one correct answer.
    Return only a **valid JSON array** in this format:
    
    [
      {{"question": "What is Python?", "options": ["A programming language", "A snake", "A car", "A fruit"], "correct_option": "A programming language"}}
    ]
    
    No extra text—only JSON!
    """

    try:
        response = ollama_client.completions.create(
            model=MODEL,
            prompt=prompt,
            max_tokens=1000,  # Increased token limit to accommodate 10 questions
            temperature=0.7
        )

        response_text = response.choices[0].text.strip()
        
        # ❌ Remove debug print
        # print("\n🔍 DEBUG: Raw Response from Llama 3:", response_text)  

        fixed_json = fix_json_structure(response_text)

        try:
            questions = json.loads(fixed_json)
            return questions if isinstance(questions, list) else "Error: Invalid JSON structure."
        except json.JSONDecodeError:
            return "Error: Unable to parse JSON after fixing."

    except Exception as e:
        return f"Error: {e}"


def conduct_quiz(skills):
    """
    Conducts an interactive quiz based on user skills.
    """
    questions = generate_mcqs(skills)
    
    if isinstance(questions, str):  # Handle errors
        print(questions)
        return

    score = 0
    print("\n--- 🧠 QUIZ STARTS ---\n")

    for i, q in enumerate(questions, start=1):
        print(f"📌 Q{i}: {q['question']}")
        for j, option in enumerate(q["options"], start=1):
            print(f"   {j}. {option}")

        while True:
            try:
                user_answer = int(input("🔹 Enter option (1-4): ")) - 1
                if 0 <= user_answer < 4:
                    break
                print("⚠️ Invalid choice! Enter a number between 1 and 4.")
            except ValueError:
                print("⚠️ Invalid input! Enter a number.")

        if q["options"][user_answer] == q["correct_option"]:
            score += 1
            print("✅ Correct!\n")
        else:
            print(f"❌ Incorrect! The correct answer is: {q['correct_option']}\n")

    print(f"\n🎉 Quiz Complete! Your Score: {score}/{len(questions)}\n")

# 🔥 Main Execution
if __name__ == "__main__":
    user_skills = input("Enter Your Skills (Comma-Separated): ").strip()
    conduct_quiz(user_skills)


Enter Your Skills (Comma-Separated):  python, java, java script, c, c++



--- 🧠 QUIZ STARTS ---

📌 Q1: What is Python?
   1. A car
   2. A fruit
   3. A snake
   4. A programming language


🔹 Enter option (1-4):  4


✅ Correct!

📌 Q2: What is Java used for?
   1. Web development only
   2. Mobile app development only
   3. Desktop application and web development
   4. Game development only


🔹 Enter option (1-4):  3


...

❌ Incorrect! The correct answer is: To increment a variable

📌 Q9: What is the primary purpose of Python in data analysis?
   1. To handle user authentication
   2. To compress images
   3. To analyze and visualize data
   4. To create a database connection


🔹 Enter option (1-4):  1


❌ Incorrect! The correct answer is: To analyze and visualize data

📌 Q10: What is the primary purpose of Java in web development?
   1. To handle user authentication
   2. To compress images
   3. To develop server-side logic
   4. To create a database connection


🔹 Enter option (1-4):  2


❌ Incorrect! The correct answer is: To develop server-side logic


🎉 Quiz Complete! Your Score: 3/10



In [28]:
import json
import re
import ipywidgets as widgets
from IPython.display import display, clear_output
from openai import OpenAI

# 🛠️ Llama 3 API Configuration
OLLAMA_API = "http://localhost:11434/v1"  # Ensure your Ollama server is running
MODEL = "llama3"

# 🔧 Initialize OpenAI client for Llama 3 via Ollama
ollama_client = OpenAI(base_url=OLLAMA_API, api_key="ollama")

# 🔧 Function to fix JSON response from Llama
def fix_json_structure(response_text):
    response_text = response_text.strip("```json").strip("```").strip()
    match = re.search(r"\[\s*{.*}\s*\]", response_text, re.DOTALL)
    if match:
        response_text = match.group(0)
    response_text = re.sub(r',\s*\]', ']', response_text)
    return response_text

# 🎯 Function to generate MCQs from Llama
def generate_mcqs(skills):
    prompt = f"""
    Generate exactly 5 multiple-choice questions (MCQs) related to {skills}.
    Each question must have 4 options and one correct answer.
    Return only a **valid JSON array** in this format:
    
    [
      {{"question": "What is Python?", "options": ["A programming language", "A snake", "A car", "A fruit"], "correct_option": "A programming language"}}
    ]
    
    No extra text—only JSON!
    """

    try:
        response = ollama_client.completions.create(
            model=MODEL,
            prompt=prompt,
            max_tokens=1000,
            temperature=0.7
        )
        response_text = response.choices[0].text.strip()
        fixed_json = fix_json_structure(response_text)
        return json.loads(fixed_json)
    
    except Exception as e:
        return f"Error: {e}"

# 🔥 Quiz State Variables
questions = []
score = 0
current_question_index = 0
output_area = widgets.Output()
next_button = widgets.Button(description="Next", layout=widgets.Layout(width='30%'))

# 📝 Function to start the quiz
def start_quiz(skills):
    global questions, score, current_question_index
    score = 0
    current_question_index = 0
    questions = generate_mcqs(skills)
    
    if isinstance(questions, str):  
        print(questions)
        return
    
    ask_question()

# 🧠 Function to ask the next question
def ask_question():
    global current_question_index, next_button
    clear_output(wait=True)
    
    if current_question_index < len(questions):
        q = questions[current_question_index]
        print(f"📌 **Q{current_question_index+1}:** {q['question']}")

        # Create answer buttons
        options_buttons = []
        for idx, option in enumerate(q["options"], 1):
            button = widgets.Button(description=f"{idx}. {option}", layout=widgets.Layout(width='50%'))
            button.on_click(lambda btn, opt=option: check_answer(opt))
            options_buttons.append(button)

        display(*options_buttons)
        display(output_area)  # Show output area for feedback

    else:
        print(f"🎉 **Quiz Complete! Your Score: {score}/{len(questions)}**")

# ✅ Function to check the answer
def check_answer(selected_option):
    global score, current_question_index, next_button
    
    correct_answer = questions[current_question_index]['correct_option']
    
    with output_area:
        clear_output(wait=True)
        if selected_option == correct_answer:
            print("✅ **Correct!** 🎉")
            score += 1
        else:
            print(f"❌ **Incorrect!** The correct answer is: {correct_answer}")

    # Display Next Button
    next_button.on_click(next_question)
    display(next_button)

# ➡️ Function to move to next question
def next_question(btn):
    global current_question_index
    current_question_index += 1
    output_area.clear_output()
    next_button.close()
    ask_question()

# 🔥 Start the Quiz
skills = input("Enter Your Skills (e.g., Python, AI, Cybersecurity): ").strip()
start_quiz(skills)


📌 **Q1:** What is the purpose of the `print()` function in Python?


Button(description='1. To read input from the user', layout=Layout(width='50%'), style=ButtonStyle())

Button(description='2. To display output to the console', layout=Layout(width='50%'), style=ButtonStyle())

Button(description='3. To calculate mathematical operations', layout=Layout(width='50%'), style=ButtonStyle())

Button(description='4. To loop through a set of statements', layout=Layout(width='50%'), style=ButtonStyle())

Output()